# Lezione 04 — Rappresentazioni sparse

**Corso:** NLP Classico con Python · **Blocco 2:** Rappresentare e classificare
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/04_representaciones_dispersas_IT.md`

## Cosa vedrai in questa lezione

- Il **sacco di parole** (*bag of words*, BoW): dal testo a un vettore di conteggi, prima **a mano** e poi con `CountVectorizer`.
- Gli **n-grammi** di parole e di caratteri: cosa recuperano e quanto fanno crescere il vocabolario.
- **TF-IDF**: calcolo a mano su un corpus minimo, la formula esatta di scikit-learn e in cosa si distingue da quella del libro di Manning e colleghi.
- La **dispersione**: matrici `scipy.sparse`, memoria ed effetto di `min_df`, `max_df` e `max_features`.
- La **similarità del coseno** e un esperimento guidato: recuperare il «gemello pulito» di ogni recensione con rumore.
- I **limiti** di queste rappresentazioni e perché il vettorizzatore si adatta (`fit`) solo ai dati di addestramento.

## Prerequisiti

- Tokenizzare con `re.findall(r"\w+", testo)` e nozioni di base di stopword, stemming e lemmatizzazione (lezioni 2 e 3).
- Python con `numpy` e `pandas`. Librerie: `scikit-learn`, `scipy`, `matplotlib`. La cella seguente installa solo ciò che manca (funziona allo stesso modo in Google Colab).
- La cartella `datos/` del corso (recensioni in spagnolo e in italiano). In Colab, caricala dal pannello dei file.

> **Come usare questo notebook:** esegui le celle in ordine. I calcoli a mano si verificano sempre con scikit-learn tramite un `assert`. Le cifre vengono da corpora piccoli e sintetici scritti per il corso (*esempio illustrativo*): non si generalizzano.

---
## Installazione

Installa solo ciò che manca. In un ambiente con tutto installato non fa nulla.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("scipy", "scipy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
print("Librerie pronte.")

---
## Cella 0 — Setup

Importiamo le librerie, fissiamo il **seme**, definiamo `cargar_datos` (cerca `datos/`, `../datos/` e `/content/datos/`) e leggiamo le 300 recensioni di ciascuna lingua. La colonna `texto` è il testo **grezzo** con rumore (maiuscole, accenti persi, refusi, emoji…) e `texto_limpio` è la sua versione pulita di riferimento.

In [ ]:
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

random.seed(42)                          # seme fisso: tutti otteniamo gli stessi risultati
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

ARCHIVO_RESENAS = {"spa": "resenas_es.csv", "ita": "resenas_it.csv"}
resenas = {idioma: pd.read_csv(cargar_datos(archivo)) for idioma, archivo in ARCHIVO_RESENAS.items()}
print({idioma: df.shape for idioma, df in resenas.items()})
print("Dati e librerie caricati correttamente.")

---
## 1. Sacco di parole a mano

Un sacco di parole descrive ogni documento con **quante volte compare ciascuna parola del vocabolario**, senza ordine. Per vederlo per intero costruiamo la matrice termini-documento di un corpus **minimo di cinque frasi** in ciascuna lingua (*esempio illustrativo*). I passi: tokenizzare, raccogliere il vocabolario ordinato e contare.

In [ ]:
# datos-es
CORPUS = {
    "spa": ["el envío llegó rápido",
           "el envío llegó tarde y muy tarde",
           "el producto no está bien",
           "el producto está muy bien",
           "el producto llegó roto"],
    "ita": ["la spedizione è arrivata veloce",
           "la spedizione è arrivata tardi e molto tardi",
           "il prodotto non è buono",
           "il prodotto è molto buono",
           "il prodotto è arrivato rotto"],
}
STOP = {"spa": ["el", "y", "muy"], "ita": ["il", "la", "e", "molto"]}

In [ ]:
def tokenizar(texto):
    return re.findall(r"\w+", texto.lower())

def bolsa_a_mano(corpus):
    """Restituisce (vocabolario ordinato, matrice di conteggi documenti × termini)."""
    tokens = [tokenizar(doc) for doc in corpus]
    vocabulario = sorted({t for doc in tokens for t in doc})
    indice = {t: i for i, t in enumerate(vocabulario)}
    matriz = np.zeros((len(corpus), len(vocabulario)), dtype=int)
    for i, doc in enumerate(tokens):
        for t, n in Counter(doc).items():
            matriz[i, indice[t]] = n
    return vocabulario, matriz

for idioma in ("spa", "ita"):
    vocabulario, matriz = bolsa_a_mano(CORPUS[idioma])
    print(f"Lingua: {idioma} · documenti: {matriz.shape[0]} · vocabolario: {matriz.shape[1]}")
    print(pd.DataFrame(matriz, columns=vocabulario, index=[f"d{i+1}" for i in range(len(matriz))]))
    print()

**Cosa osservi?** Ogni riga è un documento e ogni colonna una parola del vocabolario. `tardi` vale 2 nel documento 2 (`tardi e molto tardi`): è un **conteggio**, non uno 0/1. Nota anche che `è` (o `il`, `la`) compare in quasi tutte le frasi e che quasi tutte le celle sono zeri: qui si intuisce già la dispersione (sezione 5).

Ora verifichiamo la costruzione a mano con `CountVectorizer`. Attenzione al modello di token predefinito: scarta i token di **un solo carattere** (`y`, `e`, `è`).

In [ ]:
for idioma in ("spa", "ita"):
    corpus = CORPUS[idioma]
    vocabulario, matriz = bolsa_a_mano(corpus)

    cv = CountVectorizer(token_pattern=r"\w+")          # lo stesso criterio della nostra tokenizzazione
    X_cv = cv.fit_transform(corpus)
    assert list(cv.get_feature_names_out()) == vocabulario
    assert (X_cv.toarray() == matriz).all()

    cv_defecto = CountVectorizer()                       # modello predefinito
    cv_defecto.fit(corpus)
    perdidas = sorted(set(vocabulario) - set(cv_defecto.get_feature_names_out()))
    print(f"{idioma}: coincide con scikit-learn ✔. Con il modello predefinito il vocabolario passa da {len(vocabulario)} a {len(cv_defecto.vocabulary_)} (si perdono {perdidas})")

### I parametri di `CountVectorizer`

Quasi tutto ciò che hai fatto a mano (minuscole, modello di token, stopword) è un parametro. Confrontiamo la **dimensione del vocabolario** che risulta da ciascuna opzione sul corpus minimo. Le stopword di questo esempio sono una lista propria di tre o quattro parole (gli elenchi completi, nella lezione 3).

In [ ]:
def tamano_vocabulario(corpus, **parametros):
    parametros.setdefault("token_pattern", r"\w+")
    return len(CountVectorizer(**parametros).fit(corpus).vocabulary_)

filas = []
for etiqueta, parametros in [
    ("base (token_pattern=\\w+)", {}),
    ("stop_words (lista propria)", "stop"),
    ("min_df=2", {"min_df": 2}),
    ("max_df=0.8", {"max_df": 0.8}),
    ("max_features=5", {"max_features": 5}),
    ("ngram_range=(1, 2)", {"ngram_range": (1, 2)}),
]:
    fila = {"opzione": etiqueta}
    for idioma in ("spa", "ita"):
        p = {"stop_words": STOP[idioma]} if parametros == "stop" else parametros
        fila[idioma] = tamano_vocabulario(CORPUS[idioma], **p)
    filas.append(fila)
print(pd.DataFrame(filas).to_string(index=False))

**Cosa osservi?** `min_df=2` lascia solo le parole che compaiono in almeno due documenti (in questo corpus, 7 in spagnolo e 8 in italiano). `max_df=0.8` elimina quelle che compaiono in più dell'80 % dei documenti: qui solo `è` (in italiano, presente in tutte e cinque le frasi) e `el` in spagnolo. `ngram_range=(1, 2)` moltiplica il vocabolario per più di due. Nessuna di queste opzioni è «quella giusta»: dipendono dal compito.

---
## 2. N-grammi di parole e di caratteri

### 2.1 Bigrammi: ciò che un sacco puro perde

Le due frasi seguenti hanno **esattamente le stesse parole** e significati opposti. Per un sacco di parole sono lo stesso vettore; i bigrammi vedono la differenza.

In [ ]:
# datos-es
FRASES_OPUESTAS = {
    "spa": ("no es bueno, es malo", "es bueno, no es malo"),
    "ita": ("non è buono, è cattivo", "è buono, non è cattivo"),
}
PARES_ERRATA = {"spa": ("recomiendo", "recomeindo"), "ita": ("quelli", "qeulli")}

In [ ]:
for idioma, (a, b) in FRASES_OPUESTAS.items():
    for rango in [(1, 1), (2, 2), (1, 2)]:
        X_par = CountVectorizer(token_pattern=r"\w+", ngram_range=rango).fit_transform([a, b])
        print(f"{idioma} ngram_range={rango}: coseno = {cosine_similarity(X_par)[0, 1]:.4f}  ({X_par.shape[1]} caratteristiche)")

cv_bi = CountVectorizer(token_pattern=r"\w+", ngram_range=(2, 2)).fit(FRASES_OPUESTAS["spa"])
print("Bigrammi (spagnolo):", list(cv_bi.get_feature_names_out()))

**Cosa osservi?** Con gli unigrammi il coseno è esattamente 1 (vettori identici); con i bigrammi scende a 0,75 e con unigrammi più bigrammi a 0,909. I bigrammi catturano `non è`, `è buono`, `buono non`…: l'ordine locale. Nel corpus minimo, `non è buono` produce i bigrammi `non è` e `è buono`. **Non** risolvono la negazione in generale (`non mi è sembrato per niente buono` ha il `non` lontano da `buono`), ma recuperano parte dell'informazione di ordine.

### 2.2 N-grammi di caratteri: robustezza ai refusi

Con `analyzer="char_wb"` il vettorizzatore spezza **dentro ogni parola** (con uno spazio di riempimento ai lati). Un refuso cambia una parola intera (per il sacco di parole è un altro token), ma cambia solo **alcuni** dei suoi n-grammi di caratteri.

In [ ]:
for idioma, (correcta, errata) in PARES_ERRATA.items():
    analizador = CountVectorizer(analyzer="char_wb", ngram_range=(3, 3)).build_analyzer()
    a, b = set(analizador(correcta)), set(analizador(errata))
    X_pal = CountVectorizer(token_pattern=r"\w+").fit_transform([correcta, errata])
    X_car = CountVectorizer(analyzer="char_wb", ngram_range=(3, 3)).fit_transform([correcta, errata])
    print(f"{idioma}: {correcta!r} contro {errata!r}")
    print(f"   trigrammi di caratteri condivisi: {len(a & b)} su {len(a)} → {sorted(a & b)}")
    print(f"   coseno con parole = {cosine_similarity(X_pal)[0, 1]:.3f} · coseno con trigrammi di caratteri = {cosine_similarity(X_car)[0, 1]:.3f}")

**Cosa osservi?** Per il sacco di parole `quelli` e `qeulli` non hanno nulla in comune (coseno 0). Con i trigrammi di caratteri condividono solo `lli` e `li ` (coseno 0,333): un refuso che sposta due lettere cambia quasi tutto. In spagnolo `recomiendo` e `recomeindo` condividono sei trigrammi su dieci (`rec`, `eco`, `com`, `ndo`…): i caratteri **aiutano, non fanno miracoli**.

### 2.3 Il prezzo: il vocabolario cresce

Contiamo le caratteristiche generate da ciascuna configurazione sulle 300 recensioni **grezze** di ciascuna lingua.

In [ ]:
configuraciones = {
    "parole (1, 1)": dict(token_pattern=r"\w+"),
    "parole (1, 2)": dict(token_pattern=r"\w+", ngram_range=(1, 2)),
    "parole (1, 2), min_df=2": dict(token_pattern=r"\w+", ngram_range=(1, 2), min_df=2),
    "caratteri char_wb (3, 5)": dict(analyzer="char_wb", ngram_range=(3, 5)),
}
filas = []
for nombre, parametros in configuraciones.items():
    fila = {"configurazione": nombre}
    for idioma, df in resenas.items():
        fila[idioma] = len(CountVectorizer(**parametros).fit(df["texto"]).vocabulary_)
    filas.append(fila)
crecimiento = pd.DataFrame(filas)
print(crecimiento.to_string(index=False))

**Cosa osservi?** Con soli 300 testi, passare da parole singole a parole più bigrammi moltiplica il vocabolario per 3,4 (1 613 → 5 525 in spagnolo). I caratteri (da 3 a 5) arrivano a più di 13 000. La maggior parte di questi n-grammi compare una sola volta: `min_df=2` riduce il vocabolario (1, 2) da 5 525 a 1 009 in spagnolo. Ogni nuovo n-gramma è una colonna in più della matrice.

---
## 3. TF-IDF a mano

Un conteggio da solo sopravvaluta parole come `è` o `el`, presenti dappertutto. Il TF-IDF pesa ogni parola in base a **quanto compare nel documento (tf)** e a **quanto è rara nel corpus (idf)**.

La formula di **scikit-learn** (con `smooth_idf=True`, il valore predefinito) è:

- `idf(t) = ln((1 + n) / (1 + df(t))) + 1`, con `n` = numero di documenti e `df(t)` = numero di documenti che contengono `t`.
- `tf-idf(t, d) = tf(t, d) · idf(t)`, con `tf` = conteggio grezzo.
- Poi **ogni riga si normalizza con la norma L2** (`norm="l2"`, valore predefinito): la somma dei quadrati della riga vale 1.

La implementiamo a mano e la verifichiamo con `TfidfVectorizer`.

In [ ]:
def tfidf_a_mano(corpus):
    """tf-idf di scikit-learn: idf lisciato e righe normalizzate con L2."""
    vocabulario, conteos = bolsa_a_mano(corpus)
    n = len(corpus)
    df = (conteos > 0).sum(axis=0)                      # documenti che contengono ogni parola
    idf = np.log((1 + n) / (1 + df)) + 1
    bruto = conteos * idf                               # tf · idf
    normas = np.sqrt((bruto ** 2).sum(axis=1, keepdims=True))
    return vocabulario, conteos, df, idf, bruto, bruto / normas

for idioma in ("spa", "ita"):
    corpus = CORPUS[idioma]
    vocabulario, conteos, df, idf, bruto, tfidf = tfidf_a_mano(corpus)
    tv = TfidfVectorizer(token_pattern=r"\w+")
    X_tv = tv.fit_transform(corpus)
    assert np.allclose(X_tv.toarray(), tfidf)
    assert np.allclose(tv.idf_, idf)
    assert np.allclose(np.sqrt((tfidf ** 2).sum(axis=1)), 1.0)
    print(idioma, "coincide con TfidfVectorizer ✔")

# Tabella del documento 1 (spagnolo)
vocabulario, conteos, df, idf, bruto, tfidf = tfidf_a_mano(CORPUS["spa"])
n = len(CORPUS["spa"])
presentes = [j for j in range(len(vocabulario)) if conteos[0, j] > 0]
tabla = pd.DataFrame({
    "parola": [vocabulario[j] for j in presentes],
    "tf": [conteos[0, j] for j in presentes],
    "df": [df[j] for j in presentes],
    "idf": [round(idf[j], 4) for j in presentes],
    "tf·idf": [round(bruto[0, j], 4) for j in presentes],
    "tf-idf normalizzato (L2)": [round(tfidf[0, j], 4) for j in presentes],
})
print(f"Documento 1: {CORPUS['spa'][0]!r} (n = {n} documenti)")
print(tabla.to_string(index=False))
print("norma della riga non normalizzata:", round(float(np.sqrt((bruto[0] ** 2).sum())), 4))

**Cosa osservi?** `spedizione` compare in 2 documenti su 5: `idf = ln(6/3) + 1 = 1,6931`. `veloce`, che compare in 1: `ln(6/2) + 1 = 2,0986`. `è`, presente in tutti e cinque: `ln(6/6) + 1 = 1` (il `+ 1` evita che il suo peso sia zero). Dividendo ogni valore per la norma della riga (3,2010 per il documento 1 spagnolo) si ottiene la riga normalizzata, la cui somma dei quadrati vale 1. **Attenzione:** i valori di una riga TF-IDF **non** sono probabilità e non sommano a 1.

### La versione del libro non è quella di scikit-learn

Manning, Raghavan e Schütze (*Introduction to Information Retrieval*, cap. 6) definiscono nella loro versione più semplice `idf(t) = log(n / df(t))` e `tf-idf = tf · idf`, **senza** normalizzare le righe (il libro presenta anche altre varianti di tf e di normalizzazione). Confrontiamo i due `idf`. La differenza di base del logaritmo cambia solo la scala: qui usiamo il logaritmo naturale in entrambi.

In [ ]:
vocabulario, conteos, df, idf, bruto, tfidf = tfidf_a_mano(CORPUS["spa"])
n = len(CORPUS["spa"])
idf_libro = np.log(n / df)                              # Manning et al.: senza lisciamento, senza «+ 1»
comparacion = pd.DataFrame({"df": df, "idf scikit-learn": idf.round(4), "idf libro": idf_libro.round(4)}, index=vocabulario)
print(comparacion.T.to_string())

# Documento 1 con le due versioni
libro_d1 = conteos[0] * idf_libro
print()
print("Documento 1, versione del libro (tf · log(n/df), senza normalizzare):")
print({vocabulario[j]: round(libro_d1[j], 4) for j in range(len(vocabulario)) if conteos[0, j] > 0})
print("Documento 1, versione di scikit-learn (con L2):")
print({vocabulario[j]: round(tfidf[0, j], 4) for j in range(len(vocabulario)) if conteos[0, j] > 0})

# Senza lisciamento, scikit-learn usa ln(n/df) + 1; il nostro calcolo lo riproduce anche
tv_sin = TfidfVectorizer(token_pattern=r"\w+", smooth_idf=False).fit(CORPUS["spa"])
assert np.allclose(tv_sin.idf_, np.log(n / df) + 1)

**Cosa osservi?** Con la formula del libro, `el` vale **0** (compare in tutti i documenti, non distingue nulla) e le altre parole valgono tra 0,51 e 1,61. Con quella di scikit-learn `el` vale **1** e il resto tra 1,41 e 2,10: il `+ 1` (e il lisciamento) spostano tutti i pesi ed evitano gli zeri. Inoltre scikit-learn normalizza la riga. **Le classifiche per parola sono simili, i numeri no**: per questo non conviene confrontare valori di TF-IDF tra librerie diverse.

> **Attenzione:** questo «lisciamento dell'IDF» **non** è il lisciamento di Laplace delle lezioni 5 e 8. Qui si evita solo di dividere per zero e di azzerare il peso delle parole presenti in tutti i documenti; il lisciamento di Laplace somma pseudoconteggi perché nessun evento mai visto abbia probabilità zero.

---
## 4. Dispersione: `scipy.sparse`

Con un vocabolario di migliaia di parole, ogni recensione ne usa qualche decina: **quasi tutte le celle sono zero**. Memorizzarle una a una è uno spreco. `scipy.sparse` memorizza solo i valori non nulli; il formato **CSR** (*compressed sparse row*) usa tre array: `data` (i valori), `indices` (la colonna di ogni valore) e `indptr` (dove inizia ogni riga). Misuriamolo sulle recensioni grezze.

In [ ]:
def bytes_csr(matriz):
    return matriz.data.nbytes + matriz.indices.nbytes + matriz.indptr.nbytes

filas = []
for idioma, df in resenas.items():
    X_res = CountVectorizer(token_pattern=r"\w+").fit_transform(df["texto"])
    assert sp.issparse(X_res) and X_res.format == "csr"
    denso = X_res.toarray()
    assert denso.nbytes == X_res.shape[0] * X_res.shape[1] * X_res.dtype.itemsize
    filas.append({
        "lingua": idioma, "forma": X_res.shape, "non nulli": X_res.nnz,
        "% non nulli": round(100 * X_res.nnz / (X_res.shape[0] * X_res.shape[1]), 2),
        "CSR (KB)": round(bytes_csr(X_res) / 1024, 1), "denso (KB)": round(denso.nbytes / 1024, 1),
        "volte in meno": round(denso.nbytes / bytes_csr(X_res), 1),
    })
print(pd.DataFrame(filas).to_string(index=False))

**Cosa osservi?** Solo circa l'1 % delle celle è diverso da zero (0,93 % in italiano, 1,01 % in spagnolo) e il formato CSR occupa tra 65 e 71 volte meno memoria della matrice densa. Con 300 documenti il risparmio è di megabyte; con 100 000 documenti e un vocabolario di 100 000 parole, una matrice densa di `int64` occuperebbe circa 80 GB (100 000 × 100 000 × 8 byte) e **non ci starebbe** in un portatile: quella sparsa sì.

### Effetto di `min_df`, `max_df` e `max_features`

Questi tre parametri riducono il vocabolario. `min_df=k` scarta le parole che compaiono in meno di `k` documenti (con un numero intero) e `max_df=p` quelle che compaiono in più di una proporzione `p` di documenti. `max_features=m` conserva solo le `m` più frequenti.

In [ ]:
opciones = {
    "base": {},
    "lowercase=False": {"lowercase": False},
    "min_df=2": {"min_df": 2},
    "min_df=5": {"min_df": 5},
    "max_df=0.5": {"max_df": 0.5},
    "max_df=0.1": {"max_df": 0.1},
    "max_features=500": {"max_features": 500},
    "min_df=2, max_df=0.5": {"min_df": 2, "max_df": 0.5},
}
filas = []
for nombre, parametros in opciones.items():
    fila = {"opzione": nombre}
    for idioma, df in resenas.items():
        fila[idioma] = len(CountVectorizer(token_pattern=r"\w+", **parametros).fit(df["texto"]).vocabulary_)
    filas.append(fila)
tabla_vocab = pd.DataFrame(filas)
print(tabla_vocab.to_string(index=False))

base_es, min2_es = tabla_vocab.loc[0, "spa"], tabla_vocab.loc[2, "spa"]
print(f"\nParole che compaiono in un solo documento (spagnolo): {base_es - min2_es} su {base_es} ({100 * (base_es - min2_es) / base_es:.1f} %)")

ax = tabla_vocab.set_index("opzione")[["spa", "ita"]].plot.barh(figsize=(8, 4))
ax.invert_yaxis()
ax.set_xlabel("dimensione del vocabolario")
ax.set_title("Effetto dei parametri sul vocabolario (300 recensioni per lingua)")
plt.tight_layout()
plt.show()

**Cosa osservi?** Due parole su tre (66,2 % in spagnolo) compaiono in un solo documento: `min_df=2` riduce il vocabolario da 1 613 a 545. `max_df=0.5` non cambia quasi nulla (toglie una parola) perché in un corpus breve e vario quasi nessuna parola arriva al 50 % dei documenti; `max_df=0.1` ne toglie circa venti. Le cifre sono di questo corpus piccolo: in uno grande l'effetto di `max_df` è maggiore. `lowercase=False` **aumenta** il vocabolario (1 869): `Excelente` ed `excelente` diventano due colonne.

---
## 5. Similarità del coseno

Due documenti sono simili se i loro vettori **puntano nella stessa direzione**: `cos(u, v) = (u · v) / (‖u‖ ‖v‖)`, con valori tra 0 (nessuna parola in comune, per vettori non negativi) e 1 (stessa direzione). Poiché le righe di `TfidfVectorizer` hanno già norma 1, il coseno è semplicemente il prodotto scalare. Lo facciamo a mano con tre vettori e lo verifichiamo.

In [ ]:
def coseno(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

for idioma in ("spa", "ita"):
    corpus = CORPUS[idioma]
    tfidf_mini = TfidfVectorizer(token_pattern=r"\w+").fit_transform(corpus).toarray()
    conteos_mini = CountVectorizer(token_pattern=r"\w+").fit_transform(corpus).toarray()
    d1, d3, d4 = 0, 2, 3
    for a, b in [(d1, d3), (d3, d4)]:
        print(f"{idioma} d{a+1}-d{b+1}: a mano (TF-IDF) = {coseno(tfidf_mini[a], tfidf_mini[b]):.4f} · con conteggi = {coseno(conteos_mini[a], conteos_mini[b]):.4f}")
    matriz_cos = cosine_similarity(tfidf_mini)
    assert np.isclose(matriz_cos[d3, d4], coseno(tfidf_mini[d3], tfidf_mini[d4]))
    assert np.isclose(matriz_cos[d3, d4], tfidf_mini[d3] @ tfidf_mini[d4])          # righe con norma 1: coseno = prodotto scalare
    masparecido = np.argsort(-np.where(np.eye(5, dtype=bool), -1, matriz_cos)[d3])[0]
    print(f"   documento più simile a d3: d{masparecido + 1}")

**Cosa osservi?** `il prodotto non è buono` e `il prodotto è molto buono` (d3 e d4) sono le più simili del corpus (0,684 con TF-IDF, 0,8 con i conteggi): condividono `il`, `prodotto`, `è` e `buono`. La coincidenza di `il` pesa meno con il TF-IDF che con i conteggi. Il coseno **non distingue** `non è buono` da `è buono`: il `non` si nota solo come una parola in più.

### Recensioni simili

Sulle recensioni pulite cerchiamo, per una di esse, le tre più simili (TF-IDF di parole, adattato su tutto il corpus solo per esplorare).

In [ ]:
INDICE_CONSULTA = 5
for idioma, df in resenas.items():
    T = TfidfVectorizer(token_pattern=r"\w+").fit_transform(df["texto_limpio"])
    sim = cosine_similarity(T[INDICE_CONSULTA], T).ravel()
    sim[INDICE_CONSULTA] = -1                                  # escludiamo la recensione stessa
    print(f"[{idioma}] Consulta ({df.loc[INDICE_CONSULTA, 'etiqueta']}): {df.loc[INDICE_CONSULTA, 'texto_limpio']}")
    for j in np.argsort(-sim)[:3]:
        print(f"   {sim[j]:.3f} ({df.loc[j, 'etiqueta']}): {df.loc[j, 'texto_limpio'][:110]}")
    print()

filas = []
for idioma, df in resenas.items():
    T = TfidfVectorizer(token_pattern=r"\w+").fit_transform(df["texto_limpio"])
    S_todo = cosine_similarity(T)
    np.fill_diagonal(S_todo, -1)
    vecino = S_todo.argmax(axis=1)
    filas.append({"lingua": idioma, "stessa etichetta del suo vicino più prossimo": round((df["etiqueta"].values[vecino] == df["etiqueta"].values).mean(), 3),
                  "stessa etichetta a caso (3 classi)": round(1 / 3, 3)})
print(pd.DataFrame(filas).to_string(index=False))

**Cosa osservi?** I vicini condividono **parole**, non necessariamente significato. In spagnolo la consulta e i suoi vicini condividono la formula finale `Ahorraos el disgusto` e tutti e tre sono negativi; in italiano il vicino più prossimo della consulta positiva (un b&b) è **negativo**, perché condividono `b&b` e `mobili`. Sulle 300 recensioni, il vicino più prossimo ha la stessa etichetta il 64,3 % delle volte (italiano) e il 64,7 % (spagnolo), contro il 33,3 % del caso: la similarità lessicale informa, ma non basta.

---
## 6. Esperimento guidato: il «gemello pulito»

**Domanda.** Ogni recensione grezza (`texto`) ha una versione pulita (`texto_limpio`) che è il suo «gemello». Se indicizziamo le 300 versioni pulite e cerchiamo, per ogni testo grezzo, il più simile per coseno, quante volte recuperiamo il suo gemello?

**Ipotesi.** Gli n-grammi di caratteri sono più robusti ai refusi delle parole intere, quindi dovrebbero azzeccarne di più.

**Disegno.** Adattiamo il vettorizzatore sulle versioni pulite (l'«indice»), trasformiamo con esso i testi grezzi (consulte) e confrontiamo: TF-IDF di parole contro TF-IDF di n-grammi di caratteri (`char_wb`, da 3 a 5). Con due tipi di consulta: il **testo grezzo completo** e una consulta **corta** (solo le sue prime 3 parole), più difficile.

In [ ]:
def recuperar(df, vectorizador, consulta):
    """Per ogni consulta, indice del testo pulito più simile (il vettorizzatore si adatta ai testi puliti)."""
    indice = vectorizador.fit_transform(df["texto_limpio"])
    consultas = vectorizador.transform(consulta)
    similitudes = cosine_similarity(consultas, indice)
    return similitudes.argmax(axis=1), similitudes

def tres_primeras(texto):
    return " ".join(texto.split()[:3])

def crear(tipo):
    if tipo == "palabras":
        return TfidfVectorizer(token_pattern=r"\w+")
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))

resultados, casos = [], {}
for idioma, df in resenas.items():
    gemelo = np.arange(len(df))
    for nombre_consulta, consulta in [("testo grezzo completo", df["texto"]), ("prime 3 parole", df["texto"].map(tres_primeras))]:
        for tipo in ("palabras", "caracteres"):
            elegido, sim = recuperar(df, crear(tipo), consulta)
            aciertos = elegido == gemelo
            resultados.append({"lingua": idioma, "consulta": nombre_consulta, "TF-IDF": "parole" if tipo == "palabras" else "caratteri (3-5)",
                               "corretti (su 300)": int(aciertos.sum()),
                               "similarità minima con il gemello": round(float(sim[gemelo, gemelo].min()), 3)})
            casos[(idioma, nombre_consulta, tipo)] = (aciertos, elegido)
tabla_gemelo = pd.DataFrame(resultados)
print(tabla_gemelo.to_string(index=False))

**Cosa osservi?** Con il testo grezzo completo **entrambi azzeccano 300 su 300** in entrambe le lingue: con il testo intero gli indizi sono tanti e non c'è differenza di accuratezza. Dove i caratteri vincono davvero è nel **caso peggiore**: la similarità minima con il gemello sale da 0,568 a 0,670 (spagnolo) e da 0,644 a 0,799 (italiano). Con consulte corte (3 parole) il risultato si **mescola**: in spagnolo 244 (parole) contro 245 (caratteri); in italiano 261 contro 252, cioè **vincono le parole**. L'ipotesi si conferma solo a metà: i caratteri danno più margine di sicurezza, ma non azzeccano di più. Un esperimento con dati sintetici e testi brevi non permette di generalizzare.

Vediamo alcuni errori con la consulta corta (spagnolo, parole).

In [ ]:
idioma_ejemplo = "spa"                                   # mostriamo lo spagnolo
nombre_corta = "prime 3 parole"

filas = []
for idioma in ("spa", "ita"):
    ac_pal, _ = casos[(idioma, nombre_corta, "palabras")]
    ac_car, _ = casos[(idioma, nombre_corta, "caracteres")]
    filas.append({"lingua": idioma, "solo parole": int((ac_pal & ~ac_car).sum()), "solo caratteri": int((ac_car & ~ac_pal).sum()),
                  "entrambi": int((ac_pal & ac_car).sum()), "nessuno": int((~ac_pal & ~ac_car).sum())})
print("Consulta corta: chi azzecca")
print(pd.DataFrame(filas).to_string(index=False))

aciertos, elegido = casos[(idioma_ejemplo, nombre_corta, "palabras")]
aciertos_car, _ = casos[(idioma_ejemplo, nombre_corta, "caracteres")]
df = resenas[idioma_ejemplo]
print()
for i in np.where(~aciertos)[0][:4]:
    print(f"Consulta #{i}: {tres_primeras(df.loc[i, 'texto'])!r} · azzecca con i caratteri? {'sì' if aciertos_car[i] else 'no'}")
    print(f"   gemello : {df.loc[i, 'texto_limpio'][:90]}")
    print(f"   scelto : {df.loc[elegido[i], 'texto_limpio'][:90]}")

**Cosa osservi?** Con consulte corte, parole e caratteri sbagliano in casi **diversi**: azzeccano solo le parole 14 volte, solo i caratteri 15 (spagnolo); 18 e 9 in italiano. Nessuno dei due azzecca 41 (spagnolo) e 30 (italiano) consulte. Le cause sono due: **attacchi generici** che non distinguono (`Todo sucio y`, `Muy recomendable para`) e **refusi** dentro le tre parole (`Camara de sguridad`), che in questo esempio nemmeno i caratteri risolvono. La rappresentazione usa ciò che le dai; con poca informazione sbaglia, e i caratteri non sono una soluzione magica.

---
## 7. Limiti e una regola d'oro: `fit` solo con l'addestramento

Le rappresentazioni sparse hanno limiti chiari: il **vocabolario cresce** con il corpus, l'**ordine si perde** (salvo l'ordine locale degli n-grammi), i **sinonimi non si catturano** (`buono` ed `eccellente` sono colonne indipendenti; gli *embedding* della lezione 10 affrontano questo problema) e tutto dipende dal **preprocessing** (lezione 3): un accento perso crea una parola nuova.

Una conseguenza pratica: il vettorizzatore **impara** il suo vocabolario e i suoi `idf` con `fit`; con `transform` li applica soltanto. Se lo adatti su **tutti** i dati prima di dividere in addestramento e test, il test «si infiltra» nel modello (fuga di informazione, in dettaglio nella lezione 5). Guardiamo che vocabolario ha un `fit` fatto solo sull'80 % delle recensioni e quali parole delle recensioni di test gli sono sconosciute.

In [ ]:
filas = []
for idioma, df in resenas.items():
    entrenamiento, prueba = train_test_split(df["texto"], test_size=0.2, random_state=42)
    vec = CountVectorizer(token_pattern=r"\w+")
    vec.fit(entrenamiento)                                   # fit: solo con l'addestramento
    X_prueba = vec.transform(prueba)                         # transform: applica il vocabolario già appreso
    assert X_prueba.shape[1] == len(vec.vocabulary_)         # stesse colonne dell'addestramento

    tokens_prueba = [t for texto in prueba for t in tokenizar(texto)]
    desconocidos = [t for t in tokens_prueba if t not in vec.vocabulary_]
    filas.append({
        "lingua": idioma, "vocabolario di addestramento": len(vec.vocabulary_),
        "token di test": len(tokens_prueba), "token sconosciuti": len(desconocidos),
        "% sconosciuti": round(100 * len(desconocidos) / len(tokens_prueba), 1),
    })
print(pd.DataFrame(filas).to_string(index=False))

**Cosa osservi?** Circa un quarto dei token del test (24,0 % in italiano e 24,2 % in spagnolo) **non esistono** nel vocabolario di addestramento e `transform` li ignora: non generano nessuna colonna. È ciò che succederà con testi nuovi in produzione, ed è per questo che la valutazione va fatta così. Se adattassi il vettorizzatore su tutto il corpus, quelle parole avrebbero una colonna e un `idf`, e il risultato sarebbe ottimistico.

## 8. Riepilogo e passo successivo

- Il **sacco di parole** trasforma ogni documento in un vettore di conteggi; `CountVectorizer` lo fa con dei parametri (`lowercase`, `token_pattern`, `stop_words`, `ngram_range`, `min_df`, `max_df`, `max_features`). Il modello di token predefinito scarta i token di un carattere.
- Gli **n-grammi** recuperano ordine locale (`non è buono`) e robustezza ai refusi (caratteri), al prezzo di un vocabolario molto più grande.
- **TF-IDF di scikit-learn:** `idf = ln((1 + n) / (1 + df)) + 1` e righe con norma L2; non coincide con il `log(n/df)` del libro e **non è una probabilità**. Questo lisciamento non è quello di Laplace.
- Le matrici sono **sparse** (≈ 1 % di celle non nulle nelle recensioni) e `scipy.sparse` (CSR) le memorizza con una frazione della memoria.
- Il **coseno** confronta i documenti; con TF-IDF a righe normalizzate è un prodotto scalare. Misura parole condivise, non significato.
- Il vettorizzatore si adatta (`fit`) **solo con l'addestramento**.

**Prossimo passo:** nella lezione 5, *Classificazione di testi*, userai queste rappresentazioni dentro una `Pipeline` con Naive Bayes, regressione logistica e SVM. Prima, fai il notebook di pratica di questa lezione.